In [13]:
import numpy as np
import pandas as pd
import time

In [2]:
DATA_PATH = r"D:\AHRC\Irrigation_Git\lag_adjustment\final_output\odisha_merged_tabular.parquet"

In [3]:
FEATURE_COLS = [
    "sm_4_prior", "sm_3_prior", "sm_2_prior", "sm_1_prior",
    "sum_rainfall_4", "sum_rainfall_3", "sum_rainfall_2", "sum_rainfall_1",
    "mean_temp_4", "mean_temp_3", "mean_temp_2", "mean_temp_1",
    "doy_sin", "doy_cos",
]

In [6]:
df = pd.read_parquet(DATA_PATH)
df.columns = df.columns.str.lower()
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["lat", "lon", "date"]).reset_index(drop=True)
#df["loc_id"] = df["lat"].astype(str) + "_" + df["lon"].astype(str)
#df = df.sort_values(["loc_id", "date"]).reset_index(drop=True)
df["sm_4_prior"] = df["soil_moisture"].shift(4)
df["sm_3_prior"] = df["soil_moisture"].shift(3)
df["sm_2_prior"] = df["soil_moisture"].shift(2)
df["sm_1_prior"] = df["soil_moisture"].shift(1)
df["sum_rainfall_4"] = sum(df["rainfall"].shift(i) for i in range(1, 5))
df["sum_rainfall_3"] = sum(df["rainfall"].shift(i) for i in range(1, 4))
df["sum_rainfall_2"] = sum(df["rainfall"].shift(i) for i in range(1, 3))
df["sum_rainfall_1"] = sum(df["rainfall"].shift(i) for i in range(1, 2))
df["mean_temp_4"] = sum(df["temperature"].shift(i) for i in range(1, 5)) / 4
df["mean_temp_3"] = sum(df["temperature"].shift(i) for i in range(1, 4)) / 3
df["mean_temp_2"] = sum(df["temperature"].shift(i) for i in range(1, 3)) / 2
df["mean_temp_1"] = sum(df["temperature"].shift(i) for i in range(1, 2)) / 1
df["doy"]     = df["date"].dt.dayofyear
df["doy_sin"] = np.sin(2 * np.pi * df["doy"] / 365.25)
df["doy_cos"] = np.cos(2 * np.pi * df["doy"] / 365.25)
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
#df.dropna(subset=["sm_4_prior", "sm_3_prior", "sm_2_prior", "sm_1_prior", "sum_rainfall_4", "sum_rainfall_3", "sum_rainfall_2", "sum_rainfall_1", "mean_temp_4", "mean_temp_3", "mean_temp_2", "mean_temp_1"], inplace=True)

df = df[df["date"]>= "2025-01-01"].reset_index(drop=True) 
df["step"] = df.groupby(["lat", "lon"]).cumcount()
n_steps = df["step"].max() + 1
n_locations = df[["lat", "lon"]].drop_duplicates().shape[0]


In [7]:
full_date_range = pd.date_range(start='2025-01-01', end='2025-12-31', freq="D")
missing_reports = []
for (lat, lon), group in df.groupby(['lat', 'lon']):
    existing_dates = set(group['date'])
    missing_dates = set(full_date_range) - existing_dates
    if missing_dates != set():
        missing_reports.append({
            'lat': lat,
            'lon': lon,
            'missing_dates': sorted(missing_dates)
        })
if missing_reports == []:
    print("No missing reports found.")
    in_sync = True
else:
    print(f"Found {len(missing_reports)} locations with missing reports.")
    in_sync = False

No missing reports found.


In [10]:
print("\nNulling SM lags that fall inside test period...")
for k in [1, 2, 3, 4]:
    df.loc[df["step"] >= k, f"sm_{k}_prior"] = np.nan


Nulling SM lags that fall inside test period...


In [14]:
df["sm_pred"] = np.nan
start_time = time.time()

In [15]:
for t in range(n_steps):
    mask = df["step"] == t
    X = df.loc[mask, FEATURE_COLS].values.astype(float)
    y_pred = model.predict(X)
    df.loc[mask, "sm_pred"] = y_pred
    for offset, col in enumerate(["sm_1_prior", "sm_2_prior", "sm_3_prior", "sm_4_prior"]):
        future_step = t + offset
        if future_step >= n_steps:
            continue
        future_mask = df["step"] == future_step
        pred_series = pd.Series(
            y_pred, 
            index=pd.MultiIndex.from_frame(df.loc[mask, ["lat", "long"]]),
        )
        future_idx = pd.MultiIndex.from_frame(df.loc[future_mask, ["lat", "long"]])
        fill_vals = pred_series.reindex(future_idx).values

        # Fill only NaN slots (don't overwrite actual history values)
        current_vals = df.loc[future_mask, col]
        df.loc[future_mask, col] = current_vals.where(current_vals.notna(), fill_vals)
if (t + 1) % 10 == 0 or t == n_steps - 1:
        elapsed = time.time() - start_time
        print(f"  Step {t + 1}/{n_steps} done ({elapsed:.2f}s)")

total_time = time.time() - start_time
print(f"\nTotal prediction time: {total_time:.2f}s")



NameError: name 'model' is not defined

In [ ]:
print("\n" + "=" * 60)
print("PROPAGATION VERIFICATION (first location)")
print("=" * 60)
first_lat = df["lat"].iloc[0]
first_lon = df["long"].iloc[0]
sample = df[(df["lat"] == first_lat) & (df["long"] == first_lon)].head(8)

print(sample[[
    "date", "sm", "sm_pred",
    "sm_4_prior", "sm_3_prior", "sm_2_prior", "sm_1_prior"
]].to_string(index=False))

# Check: sm_pred of row N should equal sm_1_prior of row N+1
print("\nPropagation check (sm_pred[i] == sm_1_prior[i+1]):")
for i in range(min(5, len(sample) - 1)):
    pred_val = sample.iloc[i]["sm_pred"]
    lag_val = sample.iloc[i + 1]["sm_1_prior"]
    match = "pass" if np.isclose(pred_val, lag_val, atol=1e-6) else "FAIL"
    print(f"  Row {i} pred={pred_val:.6f}  ->  Row {i+1} sm_1_prior={lag_val:.6f}  [{match}]")


In [ ]:
valid = df.dropna(subset=["sm", "sm_pred"])
mae = mean_absolute_error(valid["sm"], valid["sm_pred"])
rmse = np.sqrt(mean_squared_error(valid["sm"], valid["sm_pred"]))
r2 = r2_score(valid["sm"], valid["sm_pred"])

print(f"  Samples : {len(valid)}")
print(f"  MAE     : {mae:.4f}")
print(f"  RMSE    : {rmse:.4f}")
print(f"  R2      : {r2:.4f}")
for t in range(min(n_steps, 20)):
    step_data = df[df["step"] == t].dropna(subset=["sm", "sm_pred"])
    if len(step_data) == 0:
        continue
    s_mae = mean_absolute_error(step_data["sm"], step_data["sm_pred"])
    s_rmse = np.sqrt(mean_squared_error(step_data["sm"], step_data["sm_pred"]))
    s_r2 = r2_score(step_data["sm"], step_data["sm_pred"]) if len(step_data) > 1 else float("nan")
    sample_date = step_data["date"].iloc[0].strftime("%Y-%m-%d")
    print(f"  {t:>5} {sample_date:>12} {len(step_data):>7} {s_mae:>8.4f} {s_rmse:>8.4f} {s_r2:>8.4f}")
